In [ ]:
PROJECT_NAME = "FSFM-Lite"
DATASET_NAME = "CelebA-Spoof"
PROTOCOL = "protocol1"
SEED = 42

print(f"Project : {PROJECT_NAME}")
print(f"Dataset : {DATASET_NAME}")
print(f"Protocol: {PROTOCOL}")

In [ ]:
import os
import json
import random

from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

random.seed(SEED)
np.random.seed(SEED)

print("Libraries Loaded Successfully")

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "CelebA_Spoof"
)

IMAGE_ROOT = (
    DATA_ROOT
    / "Data"
)

META_ROOT = (
    DATA_ROOT
    / "metas"
)

PROTOCOL_ROOT = (
    META_ROOT
    / "protocol1"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
)

METADATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "metadata"
)

METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print(PROJECT_ROOT)

In [ ]:
print("Checking Dataset Structure...\n")

required_paths = [
    DATA_ROOT,
    IMAGE_ROOT,
    META_ROOT,
    PROTOCOL_ROOT
]

for p in required_paths:
    print(
        f"{p.exists()} -> {p}"
    )

In [ ]:
TRAIN_JSON = (
    PROTOCOL_ROOT
    / "train_label.json"
)

TEST_JSON = (
    PROTOCOL_ROOT
    / "test_label.json"
)

with open(TRAIN_JSON, "r") as f:
    train_json = json.load(f)

with open(TEST_JSON, "r") as f:
    test_json = json.load(f)

print(
    "Train Samples:",
    len(train_json)
)

print(
    "Test Samples:",
    len(test_json)
)

In [ ]:
sample_key = list(train_json.keys())[0]

print("Image Path:\n")
print(sample_key)

print("\nMetadata:\n")
print(train_json[sample_key])

In [ ]:
def build_dataframe(data_dict):

    rows = []

    for img_rel_path, attrs in tqdm(data_dict.items()):

        label = 0

        if "/spoof/" in img_rel_path:
            label = 1

        img_path = (
            DATA_ROOT
            / img_rel_path
        )

        bb_path = (
            DATA_ROOT
            / img_rel_path
                .replace(".jpg", "_BB.txt")
                .replace(".png", "_BB.txt")
        )

        rows.append(
            {
                "image_rel_path": img_rel_path,
                "image_path": str(img_path),
                "bb_path": str(bb_path),
                "label": label
            }
        )

    return pd.DataFrame(rows)

In [ ]:
train_df = build_dataframe(
    train_json
)

test_df = build_dataframe(
    test_json
)

print(train_df.shape)
print(test_df.shape)

train_df.head()

In [ ]:
def verify_paths(df, n=1000):

    sample_df = df.sample(
        min(n, len(df)),
        random_state=SEED
    )

    missing_images = 0
    missing_bbox = 0

    for _, row in tqdm(sample_df.iterrows(),
                       total=len(sample_df)):

        if not Path(
            row["image_path"]
        ).exists():

            missing_images += 1

        if not Path(
            row["bb_path"]
        ).exists():

            missing_bbox += 1

    print(
        f"Missing Images: {missing_images}"
    )

    print(
        f"Missing BBoxes: {missing_bbox}"
    )

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    x=train_df["label"]
)

plt.title(
    "Train Class Distribution"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    2,
    4,
    figsize=(12,6)
)

live_df = train_df[
    train_df.label == 0
].sample(4)

spoof_df = train_df[
    train_df.label == 1
].sample(4)

for i, row in enumerate(live_df.itertuples()):

    img = Image.open(
        row.image_path
    )

    axes[0,i].imshow(img)
    axes[0,i].set_title("Live")
    axes[0,i].axis("off")

for i, row in enumerate(spoof_df.itertuples()):

    img = Image.open(
        row.image_path
    )

    axes[1,i].imshow(img)
    axes[1,i].set_title("Spoof")
    axes[1,i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
train_df.to_parquet(
    METADATA_ROOT
    / "train_df.parquet"
)

test_df.to_parquet(
    METADATA_ROOT
    / "test_df.parquet"
)

train_df.to_csv(
    METADATA_ROOT
    / "train_df.csv",
    index=False
)

test_df.to_csv(
    METADATA_ROOT
    / "test_df.csv",
    index=False
)

print("Metadata Saved")

In [ ]:
stats = {

    "train_samples":
        len(train_df),

    "test_samples":
        len(test_df),

    "train_live":
        int(
            (train_df.label==0).sum()
        ),

    "train_spoof":
        int(
            (train_df.label==1).sum()
        )
}

In [ ]:
with open(
    METADATA_ROOT
    / "dataset_stats.json",
    "w"
) as f:

    json.dump(
        stats,
        f,
        indent=4
    )

print("Dataset Stats Saved")